# F1 Telemetry Embedding Generation with Chronos-2
This notebook downloads the F1 demo dataset from Hugging Face, generates multivariate time-series embeddings using Amazon's Chronos-2 model, and visualizes the results with Renumics Spotlight.

## Download dataset from Hugging Face

In [ ]:
import datasets
import numpy as np
from renumics.spotlight.dtypes import Sequence1D

ds = datasets.load_dataset("renumics/f1_demo_dataset", split="train")
df = ds.to_pandas()

sequence_cols = ["speed", "throttle", "nGear", "brake", "drs", "x", "y", "z", "distance_driver"]
for col in sequence_cols:
    df[col] = df[col].apply(lambda s: Sequence1D(index=s[0], value=s[1]) if s is not None and len(s) == 2 else None)

df.info()
df.head()

## Visualize dataset with Spotlight

In [ ]:
from renumics import spotlight

sequence_cols = ["speed", "throttle", "nGear", "brake", "drs", "x", "y", "z", "distance_driver"]
dtype = {c: spotlight.dtypes.sequence_1d_dtype for c in sequence_cols}


spotlight.show(ds, dtype=dtype, port=8890, no_ssl=True, host="0.0.0.0", analyze=False)

## Generate embeddings with Chronos-2
We use the multivariate embedding capability of Chronos-2 to embed speed, throttle, and nGear signals jointly. Cross-variate attention captures relationships between the signals.

In [ ]:
import torch
from chronos import Chronos2Pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device)

In [ ]:
def extract_signal_values(series_col):
    """Extract y-values from Sequence1D objects."""
    signals = []
    for row in series_col:
        if row is not None and hasattr(row, "value"):
            signals.append(np.array(row.value, dtype=np.float32))
        else:
            signals.append(np.array([], dtype=np.float32))
    return signals


speed_signals = extract_signal_values(df["speed"])
throttle_signals = extract_signal_values(df["throttle"])
ngear_signals = extract_signal_values(df["nGear"])

max_len = max(
    max(len(s) for s in speed_signals),
    max(len(s) for s in throttle_signals),
    max(len(s) for s in ngear_signals),
)
print(f"Max signal length: {max_len}")

In [ ]:
def pad_signal(signal, target_len):
    """Right-pad signal with last value to target length."""
    if len(signal) == 0:
        return np.zeros(target_len, dtype=np.float32)
    if len(signal) >= target_len:
        return signal[:target_len]
    return np.pad(signal, (0, target_len - len(signal)), mode="edge")


batch_size = 8
all_embeddings = []

for i in range(0, len(df), batch_size):
    batch_tensors = []
    for j in range(i, min(i + batch_size, len(df))):
        speed_padded = pad_signal(speed_signals[j], max_len)
        throttle_padded = pad_signal(throttle_signals[j], max_len)
        ngear_padded = pad_signal(ngear_signals[j], max_len)
        stacked = np.stack([speed_padded, throttle_padded, ngear_padded])  # (3, max_len)
        batch_tensors.append(stacked)

    batch = torch.tensor(np.stack(batch_tensors), dtype=torch.float32)  # (B, 3, max_len)

    with torch.no_grad():
        embedding_list, _ = model.embed(batch)
        pooled = torch.stack([emb.mean(dim=(0, 1)) for emb in embedding_list])

    all_embeddings.append(pooled.cpu().numpy())
    print(f"Processed {min(i + batch_size, len(df))}/{len(df)} samples")

embeddings_array = np.concatenate(all_embeddings, axis=0)
print(f"Final embeddings shape: {embeddings_array.shape}")

In [ ]:
if "chronos_embedding" in ds.column_names:
    ds = ds.remove_columns("chronos_embedding")
ds = ds.add_column("chronos_embedding", [emb.tolist() for emb in embeddings_array])
print(f"Added chronos_embedding to ds: {ds.features['chronos_embedding']}")
print(f"Sample embedding length: {len(ds[0]['chronos_embedding'])}")

## Visualize embeddings with Spotlight

In [ ]:
from renumics import spotlight

sequence_cols = ["speed", "throttle", "nGear", "brake", "drs", "x", "y", "z", "distance_driver"]
dtype = {c: spotlight.dtypes.sequence_1d_dtype for c in sequence_cols}
for c in ["portrait", "speed_vis", "gear_vis"]:
    dtype[c] = spotlight.dtypes.image_dtype
embedding_cols = [
    "chronos_embedding", "speed_emb", "brake_emb", "throttle_emb", "nGear_emb",
    "brake_emb_reduced", "speed_emb_reduced", "throttle_emb_reduced",
    "nGear_emb_reduced", "chronos_embedding_reduced",
]
for c in embedding_cols:
    if c in ds.column_names:
        dtype[c] = spotlight.dtypes.embedding_dtype

spotlight.show(ds, dtype=dtype, port=8891, no_ssl=True, host="0.0.0.0", analyze=False)

## Embedding size reduction
Use UMAP to reduce high-dimensional embeddings to 2D for visualization in Spotlight's similarity map.

In [ ]:
import umap

embedding_cols = {
    "speed_emb": ds["speed_emb"],
    "throttle_emb": ds["throttle_emb"],
    "chronos_embedding": ds["chronos_embedding"],
}

# For nGear, pad signal values to uniform length as a flat embedding
ngear_raw = ds["nGear"]
ngear_max_len = max(len(row[1]) for row in ngear_raw)
ngear_padded = []
for row in ngear_raw:
    vals = np.array(row[1], dtype=np.float32)
    if len(vals) < ngear_max_len:
        vals = np.pad(vals, (0, ngear_max_len - len(vals)), mode="edge")
    ngear_padded.append(vals.tolist())
embedding_cols["nGear_emb"] = ngear_padded

for col_name, col_data in embedding_cols.items():
    arr = np.array(col_data, dtype=np.float32)
    reducer = umap.UMAP(n_components=2, random_state=42)
    reduced = reducer.fit_transform(arr)

    reduced_col = f"{col_name}_reduced"
    if reduced_col in ds.column_names:
        ds = ds.remove_columns(reduced_col)
    ds = ds.add_column(reduced_col, reduced.tolist())
    print(f"{col_name} ({arr.shape[1]}d) -> {reduced_col} (2d)")

print(f"\nDataset columns: {ds.column_names}")